# OpenRouter Free Models Router

Интерактивный smoke playbook. Конкретная бесплатная модель выбирается OpenRouter, поэтому результат не полностью воспроизводим; availability и rate limits контролируются OpenRouter.

In [ ]:
arsenal_config_path = "@comp/arsenal_openrouter_free.toml"
arsenal_stop_before_playbook_begin = False
arsenal_stop_after_playbook_end = True
prompt = "Кратко объясни, что такое ZEMI Arsenal."

## Preparation

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()

In [ ]:
from zemi.arsenal.python import PythonVenv
PythonVenv.from_config("@comp/00_init.toml").verify()

## External request

При первом запуске `OPENROUTER_API_KEY` запрашивается через `getpass` и сохраняется только в `@inst/_secrets/arsenal.env`.

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession
arsenal = ArsenalSession(arsenal_config_path)
zemi.arsenal.begin(arsenal, stop_before_begin=arsenal_stop_before_playbook_begin)
try:
    model = arsenal.endpoints.openrouter.models.openrouter_free
    assistant = model.assistants.assistant
    client = assistant.clients.openai.client
    response = client.chat.completions.create(model=assistant.clients.model, messages=[{"role": "user", "content": prompt}])
    model_response = response.choices[0].message.content
    print(model_response)
finally:
    zemi.arsenal.end(arsenal, stop_after_end=arsenal_stop_after_playbook_end)

## Output parameters

In [ ]:
from zemi.playbook import output_params
output_params({"model_response": model_response, "response_length": len(model_response)})